 # Proyecto: ucuenca-sabe

 ## Dominio: 01_docentes | Capa: Bronze -> Silver



 Este notebook automatiza la carga de los archivos crudos de GTH y Analítica (PURE),

 normaliza las identificaciones mitigando la pérdida del cero a la izquierda,

 genera una auditoría de cruces (identificando registros huérfanos) y almacena el resultado

 optimizado en la capa Silver en formato Parquet.

In [ ]:
import sys
from pathlib import Path

# Agregar src al path
sys.path.insert(0, str(Path.cwd().parent.parent / 'src'))

from config import Config
import pandas as pd
import numpy as np

## CELDA 1: Importar configuración centralizada

In [ ]:

# Verificar todo
Config.ensure_directories()
Config.verify_bronze_files()

# Usar las rutas
print(f"📁 Silver: {Config.SILVER_DIR}")
print(f"📁 Gold: {Config.GOLD_DIR}")

# Cargar archivos
df_gth = pd.read_excel(Config.GTH_FILE, dtype=str)
df_andat = pd.read_excel(Config.ANALITICA_FILE, dtype=str)

print(f"✅ GTH cargado: {df_gth.shape}")
print(f"✅ Analítica cargado: {df_andat.shape}")

In [ ]:
# ==========================================
# CELDA 2: EXPLORACIÓN BÁSICA DE GTH
# ==========================================
print("="*80)
print("📊 EXPLORACIÓN GTH - MATRIZ_ENVIADA_2")
print("="*80)

# Info básica
print(f"Filas: {df_gth.shape[0]}")
print(f"Columnas: {df_gth.shape[1]}")
print(f"\nColumnas y longitud de valores:")
for col in df_gth.columns:
    nulos = df_gth[col].isnull().sum()
    # Longitud máxima y mínima de los valores
    longitudes = df_gth[col].dropna().str.len()
    if len(longitudes) > 0:
        print(f"  • {col}: min={longitudes.min()}, max={longitudes.max()}, nulos={nulos}")
    else:
        print(f"  • {col}: SIN DATOS, nulos={nulos}")

print(f"\nPrimeras 3 filas:")
display(df_gth.head(3))

In [ ]:
# ==========================================
# CELDA 3: EXPLORACIÓN BÁSICA DE ANALITICA
# ==========================================
print("="*80)
print("📊 EXPLORACIÓN ANALITICA - docentes_titulo_unesco")
print("="*80)
# Info básica
print(f"Filas: {df_andat.shape[0]}")
print(f"Columnas: {df_andat.shape[1]}")
print(f"\nColumnas y longitud de valores:")
for col in df_andat.columns:
    nulos = df_andat[col].isnull().sum()
    # Longitud máxima y mínima de los valores
    longitudes = df_andat[col].dropna().str.len()
    if len(longitudes) > 0:
        print(f"  • {col}: min={longitudes.min()}, max={longitudes.max()}, nulos={nulos}")
    else:
        print(f"  • {col}: SIN DATOS, nulos={nulos}")

print(f"\nPrimeras 3 filas:")
display(df_andat.head(3))

In [ ]:
# ==========================================
# CELDA 2: CARGAR Y NORMALIZAR (UNA SOLA VEZ)
# ==========================================
print("="*80)
print("📊 CARGA Y NORMALIZACIÓN DE CÉDULAS")
print("="*80)

print(f"GTH: {df_gth.shape[0]} filas")
print(f"ANALITICA: {df_andat.shape[0]} filas")

# Columnas de cédula
col_gth = "CEDULA"
col_andat = "NUMERO_DOCUMENTO"

# FUNCIÓN ÚNICA de normalización
def normalizar_cedula(serie):
    return (serie.astype(str)
                 .str.strip()
                 .str.replace(r'[^0-9]', '', regex=True)
                 .str.zfill(10))

# Aplicar normalización UNA SOLA VEZ
df_gth['CEDULA_NORM'] = normalizar_cedula(df_gth[col_gth])
df_andat['CEDULA_NORM'] = normalizar_cedula(df_andat[col_andat])

print(f"\n✅ Cédulas normalizadas en GTH: {df_gth['CEDULA_NORM'].nunique()} únicas")
print(f"✅ Cédulas normalizadas en ANALITICA: {df_andat['CEDULA_NORM'].nunique()} únicas")

# Mostrar ejemplos
print(f"\nEjemplos GTH: {df_gth['CEDULA_NORM'].head(5).tolist()}")
print(f"Ejemplos ANALITICA: {df_andat['CEDULA_NORM'].head(5).tolist()}")

Dado que analítica de las 1142 filas , solo 1136 son unicos y al analizar vemos que son duplicados, parecen ser- 

In [ ]:
# ==========================================
# CELDA: ANÁLISIS DE DUPLICADOS EN ANALITICA
# ==========================================
print("="*80)
print("🔍 ANÁLISIS DE DUPLICADOS - ANALITICA")
print("="*80)

col_andat = "NUMERO_DOCUMENTO"

print(f"Total filas: {len(df_andat)}")
print(f"Cédulas únicas: {df_andat[col_andat].nunique()}")
print(f"Diferencia: {len(df_andat) - df_andat[col_andat].nunique()} duplicados\n")

# Encontrar cédulas duplicadas
duplicados = df_andat[df_andat.duplicated(subset=col_andat, keep=False)]
cedulas_dup = duplicados[col_andat].unique()

print(f"Cédulas que aparecen más de una vez: {len(cedulas_dup)}")
print(f"\nCédulas duplicadas:")
for ced in cedulas_dup:
    count = (df_andat[col_andat] == ced).sum()
    print(f"  • {ced}: aparece {count} veces")

# Mostrar todas las filas duplicadas completas
print(f"\n📋 Registros duplicados completos ({len(duplicados)} filas):")
display(duplicados.sort_values(by=col_andat))

In [ ]:
# ==========================================
# CELDA 2.5: PIVOTEAR DUPLICADOS SIN PERDER COLUMNAS
# ==========================================
print("🔄 PIVOTEANDO DUPLICADOS DE ANALITICA (CONSERVANDO TODAS LAS COLUMNAS)")
print("="*80)

# Columnas que se expanden (las que varían por docente)
columnas_a_pivotear = ['UNIDAD ACADÉMICA', 'TIPO_DEDICACION']

# Columnas que NO varían (las mantenemos igual)
# Son todas menos CEDULA_NORM, orden_temp, fila_num, y las pivotadas
cols_fijas = [col for col in df_andat.columns 
              if col not in columnas_a_pivotear + ['CEDULA_NORM', 'orden_temp', 'fila_num']]

print(f"Columnas fijas (primera ocurrencia): {cols_fijas}")
print(f"Columnas a expandir: {columnas_a_pivotear}")

# 1. Ordenar para que TIEMPO COMPLETO quede primero
orden_dedicacion = {'TIEMPO COMPLETO': 0, 'MEDIO TIEMPO': 1, 'TIEMPO PARCIAL': 2, 'POR HORAS': 3}
df_andat['orden_temp'] = df_andat['TIPO_DEDICACION'].map(orden_dedicacion).fillna(99)

# 2. Ordenar por cédula y prioridad
df_andat_sorted = df_andat.sort_values(['CEDULA_NORM', 'orden_temp'])

# 3. Contador por cédula
df_andat_sorted['fila_num'] = df_andat_sorted.groupby('CEDULA_NORM').cumcount() + 1

# 4. Pivotear SOLO las columnas que varían
df_pivot = df_andat_sorted.pivot_table(
    index='CEDULA_NORM',
    columns='fila_num',
    values=columnas_a_pivotear,
    aggfunc='first'
)

# 5. Aplanar columnas pivotadas
df_pivot.columns = [f'{col[0]}_{int(col[1])}' for col in df_pivot.columns]
df_pivot = df_pivot.reset_index()

# 6. Tomar la PRIMERA fila de cada cédula para las columnas fijas
df_fijas = df_andat_sorted.groupby('CEDULA_NORM')[cols_fijas].first().reset_index()

# 7. Unir pivot + fijas
df_andat_pivot = df_pivot.merge(df_fijas, on='CEDULA_NORM', how='left')

print(f"\n✅ ANALITICA pivotado (COMPLETO):")
print(f"   Antes: {len(df_andat)} filas, {len(df_andat.columns)} columnas")
print(f"   Ahora: {len(df_andat_pivot)} filas, {len(df_andat_pivot.columns)} columnas")
print(f"\n📋 TODAS las columnas:")
for i, col in enumerate(df_andat_pivot.columns, 1):
    print(f"   {i:2d}. {col}")

# Verificar
print(f"\n🔍 Cédulas únicas: {df_andat_pivot['CEDULA_NORM'].nunique()} de {len(df_andat_pivot)} filas")
print(f"📋 Ejemplo:")
display(df_andat_pivot.head(2))

In [ ]:
# CELDA 3: CHEQUEO DE MATCH (con df_andat_pivot COMPLETO)

print(" CHEQUEO DE MATCH")


set_gth = set(df_gth['CEDULA_NORM'].dropna())
set_andat = set(df_andat_pivot['CEDULA_NORM'].dropna())  # ← USAR df_andat_pivot

ambas = set_gth & set_andat
solo_gth = set_gth - set_andat
solo_andat = set_andat - set_gth

print(f"""
📊 RESULTADOS:
   ✅ En ambas bases:  {len(ambas)} docentes
   ⚠️ Solo en GTH:     {len(solo_gth)} docentes
   ⚠️ Solo en ANALITICA: {len(solo_andat)} docentes
   ─────────────────────────────
   📁 Total GTH:       {len(set_gth)} cédulas únicas
   📁 Total ANALITICA: {len(set_andat)} cédulas únicas
""")

In [ ]:
# ==========================================
# CELDA 4: OUTER JOIN (usando df_andat_pivot)
# ==========================================
print("="*80)
print("🔗 OUTER JOIN: CONSERVANDO TODOS LOS DOCENTES")
print("="*80)

# OUTER JOIN con la versión pivotada
df_full = df_gth.merge(
    df_andat_pivot,
    how="outer",
    on='CEDULA_NORM',
    indicator=True,
    suffixes=("_GTH", "_ANDAT")
)

print(f"\n📊 RESULTADO DEL OUTER JOIN:")
print(f"   Total docentes: {len(df_full)}")
print(f"\n   ✅ both:        {(df_full['_merge']=='both').sum()} (en ambas bases)")
print(f"   ⚠️ left_only:    {(df_full['_merge']=='left_only').sum()} (solo GTH)")
print(f"   ⚠️ right_only:   {(df_full['_merge']=='right_only').sum()} (solo ANALITICA)")

print(f"\n📋 Primeros 3 registros:")
display(df_full.head(3))

In [ ]:
# ==========================================
# CELDA 5: CONSERVAR AMBAS BASES + FLAG DE ORIGEN
# ==========================================
print("="*80)
print("CONSERVANDO TODOS LOS DOCENTES (match y no-match)")
print("="*80)

# Ya no filtramos a "both": guardamos el outer join completo.
# _merge nos dice de donde vino cada fila -> lo convertimos en columna legible.
df_match = df_full.copy()
df_match['ORIGEN_MATCH'] = df_match['_merge'].map({
    'both': 'ambas_bases',
    'left_only': 'solo_gth',
    'right_only': 'solo_analitica',
})
df_match = df_match.drop(columns=['_merge'])

# Identificar columnas de cedula a eliminar
columnas_cedula_extra = ['CEDULA', 'CEDULA_GTH', 'CEDULA_ANDAT', 'TIPO_DOCUMENTO', 'NUMERO_DOCUMENTO', 'NUMERO_DOCUMENTO_GTH', 'NUMERO_DOCUMENTO_ANDAT']

# Ver cuales existen realmente en df_match
columnas_a_eliminar = [col for col in columnas_cedula_extra if col in df_match.columns]

print("\nEliminando columnas de cedula redundantes:")
for col in columnas_a_eliminar:
    print(f"   - {col}")

# Eliminar esas columnas
df_match = df_match.drop(columns=columnas_a_eliminar, errors='ignore')

print("\nRESULTADO FINAL:")
print(f"   Total: {len(df_match)} docentes")
print(df_match['ORIGEN_MATCH'].value_counts())
print(f"   Columnas: {len(df_match.columns)}")

# Verificar que CEDULA_NORM esta presente
if 'CEDULA_NORM' in df_match.columns:
    print("   OK: CEDULA_NORM conservada")
else:
    print("   FALTA: CEDULA_NORM no encontrada")

# Verificar duplicados
print("\nVerificacion:")
print(f"   Cedulas unicas: {df_match['CEDULA_NORM'].nunique()}")
print(f"   Filas totales: {len(df_match)}")
print(f"   Duplicados: {len(df_match) - df_match['CEDULA_NORM'].nunique()}")

# Mostrar TODAS las columnas finales
print(f"\nColumnas finales ({len(df_match.columns)}):")
for i, col in enumerate(df_match.columns, 1):
    print(f"   {i:2d}. {col}")

# Mostrar primeras filas
print("\nPrimeros 3 registros:")
display(df_match.head(3))

# Info rapida de nulos
print("\nNulos por columna:")
nulos = df_match.isnull().sum()
nulos_pct = (nulos / len(df_match)) * 100
for col in df_match.columns:
    if nulos[col] > 0:
        print(f"   - {col}: {nulos[col]} nulos ({nulos_pct[col]:.1f}%)")


In [ ]:
# Guardar el catastro limpio usando TU constante actual
df_match.to_parquet(Config.CATRASTRO_DOCENTES, index=False, engine='pyarrow')
print(f"✅ Catastro guardado en: {Config.CATRASTRO_DOCENTES}")

print("\n==================================================")
print("🏆 PIPELINE DE BIBLIOTECA FINALIZADO EN CAPA SILVER")
print("==================================================\n")